# Bronze Layer — Configurable MIMIC-IV Ingestion

## Data flow
`AWS S3` → `Unity Catalog external volume` → `configurable Bronze ingestion` → `Delta Bronze tables`

## Purpose
One ingestion entry point for the 31 expected MIMIC-IV tables (22 `hosp`, 9 `icu`) — all 31 in
production scope. 

| `SOURCE_MODE` | Source | Layout | Bronze target |
|---|---|---|---|
| `aws_volume` | AWS-backed external volume `/Volumes/cdr_mimiciv/bronze/s3_mimic_iv` (production / full data) | `hosp/` and `icu/` subfolders | `cdr_mimiciv.bronze` |
| `demo` | MIMIC-IV demo files uploaded to a volume | `hosp/` and `icu/` subfolders | `cdr_mimiciv.bronze_demo` |
| `upload` | Files uploaded through the Databricks UI | flat folder | `cdr_mimiciv.bronze_upload` |

Only `aws_volume` is intended for the full dataset. The `demo` and `upload` modes are
supported by configuration but use their own schemas so they can never mix with the full data.

## Bronze principles
- Source-aligned: the Bronze schema contains only MIMIC source tables, with source
  columns and values unchanged. Empty CSV fields load as NULL (Spark CSV default).
- Create only missing tables. Existing tables are skipped.
- Tables in `EXCLUDED_TABLES` are never created, rebuilt or counted as trusted coverage. The set
  is empty today; every registry table is in scope.
- Tables are never deleted. Nothing is overwritten unless `REBUILD = True` **and** the table is named in `REBUILD_CONFIRM`.
- `.csv.gz` is preferred over `.csv`. A duplicate in the same folder is ignored and reported.
- Parsing is strict (`FAILFAST`): malformed input stops the load instead of silently changing values.

## Validation without repeated scans
Compressed files cannot be split, so every scan of a large file is expensive. Each table is
scanned twice, not three times:
1. **Scan 1** — schema inference (`inferSchema`, the existing Bronze pattern).
2. **Scan 2** — parse, write and count in the same job. The source row count is captured
   *while writing* with `DataFrame.observe()`, so no separate re-read is needed.

The Bronze row count is then read from the Delta table (answered from file statistics, not by
re-reading the source) and compared with the observed source count.

## Operational evidence
One row per table per run is appended to `cdr_mimiciv.default.ingestion_run_log` (run id, user,
mode, source file and size, action, source and Bronze row counts, status). The Bronze schema is
never used for operational tables.

## How to run
1. Set `SOURCE_MODE` (and optionally `TABLES`) in the configuration cell.
2. Run with `DRY_RUN = True` (default). It lists sources and checks which Bronze tables exist, and writes nothing.
3. Review the plan, then set `DRY_RUN = False` to create the missing tables.
4. Optional: `CHECK_CREATE_GUARD = True` re-attempts the create on the first table created in the run and
   confirms it is refused and the table is unchanged. New tables use `writeTo(...).using("delta").create()`,
   which fails if the table already exists and never replaces it.

## Known caveats
- Bronze tables created before this version were loaded with Spark CSV defaults. This version
  adds `multiLine` and RFC-4180 quote escaping; files without embedded newlines or doubled
  quotes parse identically.
- Inference chooses column types from the data, as before. Types therefore follow the file content.

## Known limitations and data-quality decisions
- **Decision:** the existing Bronze copies `cdr_mimiciv.bronze.poe` and
  `cdr_mimiciv.bronze.poe_detail` are accepted as the source of record for the orders domain. They
  feed Silver `poe` / `poe_detail` and the Gold orders tables (`dim_order_type`, `fact_order`,
  `fact_order_detail`).
- **No rebuild.** This notebook does not re-ingest them and does not need to: both tables exist, so
  a normal run skips them. Rebuilding would need `REBUILD = True` **and** the table named in
  `REBUILD_CONFIRM`, and should not be attempted until the provenance question below is settled.
- **Reporting:** coverage is reported as expected / excluded / present / missing. Nothing is
  excluded today, so the trusted pipeline covers all 31 expected MIMIC-IV tables.

## 1. Configuration

In [0]:
SOURCE_MODE = "aws_volume"          # "aws_volume" | "demo" | "upload"

SOURCES = {
    "aws_volume": {
        "root": "/Volumes/cdr_mimiciv/bronze/s3_mimic_iv",
        "layout": "module_subfolders",          # <root>/hosp/<table>.csv.gz, <root>/icu/<table>.csv.gz
        "target": "cdr_mimiciv.bronze",
    },
    "demo": {
        # set to the volume folder that contains hosp/ and icu/ once the demo files are uploaded
        "root": "/Volumes/cdr_mimiciv/bronze_demo/mimic_iv_demo",
        "layout": "module_subfolders",
        "target": "cdr_mimiciv.bronze_demo",
    },
    "upload": {
        "root": "/Volumes/cdr_mimiciv/default/raw",
        "layout": "flat",                       # <root>/<table>.csv.gz
        "target": "cdr_mimiciv.bronze_upload",
    },
}

TABLES = "all"                      # "all" or a list of registry table names
REBUILD = False                     # default: create only missing tables
REBUILD_CONFIRM = []                # a rebuild also requires the table names to be listed here
DRY_RUN = True                      # True: print the plan only, write nothing
VALIDATE_ROW_COUNTS = True          # compare observed source rows with Bronze rows for every table
CHECK_CREATE_GUARD = False          # optional regression check: CREATE must refuse an existing table

# Expected MIMIC tables that are intentionally NOT part of the trusted production pipeline. Such a
# table is never created, rebuilt, deleted or counted as trusted coverage here, and an existing Bronze
# copy is left untouched and reported as untrusted.
EXCLUDED_TABLES = {}

FILE_PREFERENCE = ["csv.gz", "csv"]  # first match wins; later matches in the same folder are ignored
CSV_OPTIONS = {
    "header": "true",
    "inferSchema": "true",           # existing Bronze pattern
    "mode": "FAILFAST",              # malformed records stop the load
    "multiLine": "true",             # quoted fields may contain line breaks
    "escape": '"',                   # RFC 4180: a quote inside a quoted field is written as ""
}

RUN_LOG_TABLE = "cdr_mimiciv.default.ingestion_run_log"

# other places where copies of the same files are known to exist; reported only, never used
OTHER_SOURCE_LOCATIONS = [
    "/Volumes/cdr_mimiciv/default/raw",
    "/Volumes/cdr_mimiciv/bronze/files",
]

## 2. Table registry
The single list of the 31 expected MIMIC-IV tables and the module each belongs to.

In [0]:
TABLE_REGISTRY = {
    # hosp (22)
    "admissions": "hosp",
    "d_hcpcs": "hosp",
    "d_icd_diagnoses": "hosp",
    "d_icd_procedures": "hosp",
    "d_labitems": "hosp",
    "diagnoses_icd": "hosp",
    "drgcodes": "hosp",
    "emar": "hosp",
    "emar_detail": "hosp",
    "hcpcsevents": "hosp",
    "labevents": "hosp",
    "microbiologyevents": "hosp",
    "omr": "hosp",
    "patients": "hosp",
    "pharmacy": "hosp",
    "poe": "hosp",
    "poe_detail": "hosp",
    "prescriptions": "hosp",
    "procedures_icd": "hosp",
    "provider": "hosp",
    "services": "hosp",
    "transfers": "hosp",
    # icu (9)
    "caregiver": "icu",
    "chartevents": "icu",
    "d_items": "icu",
    "datetimeevents": "icu",
    "icustays": "icu",
    "ingredientevents": "icu",
    "inputevents": "icu",
    "outputevents": "icu",
    "procedureevents": "icu",
}

assert len(TABLE_REGISTRY) == 31, "registry must hold exactly the 31 MIMIC-IV tables"
assert sum(m == "hosp" for m in TABLE_REGISTRY.values()) == 22
assert sum(m == "icu" for m in TABLE_REGISTRY.values()) == 9

## 3. Configuration checks and source plan

In [0]:
import uuid
from datetime import datetime, timezone

from pyspark.sql import Observation, functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, BooleanType, TimestampType
)

if SOURCE_MODE not in SOURCES:
    raise ValueError(f"SOURCE_MODE must be one of {list(SOURCES)}")

SOURCE = SOURCES[SOURCE_MODE]
SOURCE_ROOT = SOURCE["root"].rstrip("/")
LAYOUT = SOURCE["layout"]
TARGET = SOURCE["target"]

selected = list(TABLE_REGISTRY) if TABLES == "all" else list(TABLES)
unknown = [t for t in selected if t not in TABLE_REGISTRY]
if unknown:
    raise ValueError(f"Not in the registry: {unknown}")

if REBUILD and not REBUILD_CONFIRM:
    raise ValueError("REBUILD = True requires the table names to be listed in REBUILD_CONFIRM")
bad_confirm = [t for t in REBUILD_CONFIRM if t not in TABLE_REGISTRY]
if bad_confirm:
    raise ValueError(f"REBUILD_CONFIRM names not in the registry: {bad_confirm}")
bad_excluded = [t for t in EXCLUDED_TABLES if t not in TABLE_REGISTRY]
if bad_excluded:
    raise ValueError(f"EXCLUDED_TABLES names not in the registry: {bad_excluded}")
excluded_rebuild = [t for t in REBUILD_CONFIRM if t in EXCLUDED_TABLES]
if excluded_rebuild:
    raise ValueError(f"Excluded tables cannot be rebuilt: {excluded_rebuild}")
rebuild_set = set(REBUILD_CONFIRM) if REBUILD else set()

print(f"SOURCE_MODE: {SOURCE_MODE}")
print(f"Source root: {SOURCE_ROOT}  (layout: {LAYOUT})")
print(f"Target     : {TARGET}")
print(f"Tables     : {len(selected)} selected | REBUILD={REBUILD} {sorted(rebuild_set)} | DRY_RUN={DRY_RUN}")
print(f"Excluded   : {sorted(EXCLUDED_TABLES) if EXCLUDED_TABLES else 'none'}")

In [0]:
def list_directory(path):
    """name -> size for the files in a folder, or None if the folder cannot be listed."""
    try:
        return {f.name.rstrip("/"): f.size for f in dbutils.fs.ls(path)}
    except Exception:
        return None


def mb(n):
    return f"{n / 1e6:,.1f} MB"


def build_plan():
    listings = {}
    plan, warnings, other_copies = [], [], {}

    def listing_for(directory):
        if directory not in listings:
            listings[directory] = list_directory(directory)
            if listings[directory] is None:
                warnings.append(f"Source folder not found or not readable: {directory}")
        return listings[directory]

    other_listings = {loc: list_directory(loc) for loc in OTHER_SOURCE_LOCATIONS}

    for table in selected:
        module = TABLE_REGISTRY[table]
        directory = f"{SOURCE_ROOT}/{module}" if LAYOUT == "module_subfolders" else SOURCE_ROOT
        listing = listing_for(directory) or {}

        present = [f"{table}.{ext}" for ext in FILE_PREFERENCE if f"{table}.{ext}" in listing]
        source_file = f"{directory}/{present[0]}" if present else None
        source_bytes = listing[present[0]] if present else None

        for ignored in present[1:]:
            warnings.append(
                f"{table}: duplicate '{ignored}' ({mb(listing[ignored])}) in the source folder is "
                f"ignored; '{present[0]}' is used"
            )

        # copies elsewhere are never used; a same-named file of a different size is worth a warning
        for loc, other in other_listings.items():
            if not other or loc == directory:
                continue
            if present and present[0] in other and other[present[0]] != source_bytes:
                warnings.append(
                    f"{table}: a different-sized copy of '{present[0]}' exists at {loc} "
                    f"({mb(other[present[0]])} vs {mb(source_bytes)} used); not used"
                )
            elif any(f"{table}.{ext}" in other for ext in FILE_PREFERENCE):
                other_copies.setdefault(loc, []).append(table)

        exists = spark.catalog.tableExists(f"{TARGET}.{table}")
        if table in EXCLUDED_TABLES:
            action = "EXCLUDED"                    # wins over every other setting
        elif source_file is None:
            action = "NO_SOURCE"
        elif exists and table in rebuild_set:
            action = "REBUILD"
        elif exists:
            action = "SKIP_EXISTS"
        else:
            action = "CREATE"

        plan.append({
            "table": table, "module": module, "target_table": f"{TARGET}.{table}",
            "source_file": source_file, "source_bytes": source_bytes, "action": action,
            "exists": exists,
        })

    for loc, tables in other_copies.items():
        warnings.append(f"{len(tables)} tables also have another copy or format at {loc} (not used)")

    # report files in the source folders that are not part of the registry
    for directory, listing in listings.items():
        if not listing:
            continue
        expected = {f"{t}.{ext}" for t in TABLE_REGISTRY for ext in FILE_PREFERENCE}
        extra = sorted(n for n in listing if n not in expected)
        if extra:
            warnings.append(f"Unregistered files in {directory} (ignored): {extra}")

    return plan, warnings


def coverage_report(plan, exists, ingested=()):
    """Expected / excluded / present / missing, so excluded tables never count as trusted coverage."""
    excluded = [p["table"] for p in plan if p["action"] == "EXCLUDED"]
    scope = [p["table"] for p in plan if p["action"] != "EXCLUDED"]
    present = [t for t in scope if exists[t]]
    missing = [t for t in scope if not exists[t]]
    untrusted = [t for t in excluded if exists[t]]
    print(f"\nCoverage in {TARGET}")
    print(f"  expected MIMIC source tables : {len(plan)}")
    print(f"  intentionally excluded       : {len(excluded)} {excluded}")
    if untrusted:
        print(f"    existing Bronze copies left untouched and NOT trusted: {untrusted}")
    print(f"  trusted pipeline scope       : {len(scope)} of {len(plan)}")
    print(f"  present in scope             : {len(present)}  (ingested in this run: {len(list(ingested))})")
    print(f"  missing in scope             : {len(missing)} {missing}")


def print_plan(plan, warnings):
    print(f"{'table':<20}{'module':<7}{'action':<13}{'size':>12}  source file")
    for p in sorted(plan, key=lambda x: (x["action"], x["source_bytes"] or 0)):
        size = mb(p["source_bytes"]) if p["source_bytes"] is not None else "-"
        note = "  [existing Bronze copy NOT trusted]" if p["action"] == "EXCLUDED" and p["exists"] else ""
        print(f"{p['table']:<20}{p['module']:<7}{p['action']:<13}{size:>12}  {p['source_file'] or '-'}{note}")
    counts = {}
    for p in plan:
        counts[p["action"]] = counts.get(p["action"], 0) + 1
    print("\nSummary:", counts)
    create = [p["table"] for p in plan if p["action"] == "CREATE"]
    print(f"Would create ({len(create)}): {create}")
    coverage_report(plan, {p["table"]: p["exists"] for p in plan})
    if warnings:
        print("\nWarnings:")
        for w in warnings:
            print(" -", w)


plan, warnings = build_plan()
print_plan(plan, warnings)

## 4. Ingestion
Runs only when `DRY_RUN = False`. A dry run stops after the plan above and writes nothing.

In [0]:
LOG_SCHEMA = StructType([
    StructField("run_id", StringType()),
    StructField("run_started_at", TimestampType()),
    StructField("run_user", StringType()),
    StructField("source_mode", StringType()),
    StructField("source_root", StringType()),
    StructField("target_schema", StringType()),
    StructField("table_name", StringType()),
    StructField("module", StringType()),
    StructField("source_file", StringType()),
    StructField("source_bytes", LongType()),
    StructField("action", StringType()),
    StructField("status", StringType()),
    StructField("source_rows", LongType()),
    StructField("bronze_rows", LongType()),
    StructField("rows_match", BooleanType()),
    StructField("message", StringType()),
    StructField("started_at", TimestampType()),
    StructField("completed_at", TimestampType()),
])


def now():
    return datetime.now(timezone.utc)


def ensure_log_table():
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {RUN_LOG_TABLE} (
            run_id STRING, run_started_at TIMESTAMP, run_user STRING, source_mode STRING,
            source_root STRING, target_schema STRING, table_name STRING, module STRING,
            source_file STRING, source_bytes BIGINT, action STRING, status STRING,
            source_rows BIGINT, bronze_rows BIGINT, rows_match BOOLEAN, message STRING,
            started_at TIMESTAMP, completed_at TIMESTAMP
        ) USING DELTA
    """)


def write_log(run, p, status, started_at, source_rows=None, bronze_rows=None, message=None):
    rows_match = None if source_rows is None or bronze_rows is None else source_rows == bronze_rows
    row = (
        run["run_id"], run["started_at"], run["user"], SOURCE_MODE, SOURCE_ROOT, TARGET,
        p["table"], p["module"], p["source_file"], p["source_bytes"], p["action"], status,
        source_rows, bronze_rows, rows_match, message, started_at, now(),
    )
    (spark.createDataFrame([row], LOG_SCHEMA)
          .write.format("delta").mode("append").saveAsTable(RUN_LOG_TABLE))


def ingest_table(run, p):
    started_at = now()
    table = p["table"]
    try:
        # scan 1: schema inference
        df = spark.read.options(**CSV_OPTIONS).csv(p["source_file"])

        # scan 2: parse + write + count in one job; rows are counted as they are written
        source_count = Observation(f"source_rows_{table}")
        counted = df.observe(source_count, F.count(F.lit(1)).alias("rows"))

        if p["action"] == "REBUILD":
            # the only overwrite path: needs REBUILD = True and the table named in REBUILD_CONFIRM
            if not (REBUILD and table in REBUILD_CONFIRM):
                raise RuntimeError(f"{table}: rebuild requested without REBUILD = True and confirmation")
            writer = counted.write.format("delta")
            writer = writer.mode("overwrite").option("overwriteSchema", "true")
            writer.saveAsTable(p["target_table"])
        elif p["action"] == "CREATE":
            # create-only Unity Catalog write: fails if the table already exists, never replaces it
            counted.writeTo(p["target_table"]).using("delta").create()
        else:
            raise RuntimeError(f"{table}: unexpected action '{p['action']}' for ingestion")

        source_rows = source_count.get["rows"]
        bronze_rows = spark.table(p["target_table"]).count()   # Delta statistics, no source re-read

        if VALIDATE_ROW_COUNTS and source_rows != bronze_rows:
            write_log(run, p, "VALIDATION_FAILED", started_at, source_rows, bronze_rows,
                      "source and Bronze row counts differ")
            return "VALIDATION_FAILED"

        write_log(run, p, "SUCCESS", started_at, source_rows, bronze_rows)
        print(f"  {table}: {source_rows:,} source rows -> {bronze_rows:,} Bronze rows")
        return "SUCCESS"
    except Exception as e:
        write_log(run, p, "FAILED", started_at, message=str(e)[:2000])
        print(f"  {table}: FAILED - {str(e)[:300]}")
        return "FAILED"


def check_create_guard(p):
    """Regression check: the create-only write must fail, not replace, when the table already exists.

    Runs against a table created earlier in this same run, re-reading the identical source, so a broken
    guard could only replace the table with identical content - and this check would then fail loudly.
    """
    target = p["target_table"]
    version_before = spark.sql(f"DESCRIBE HISTORY {target} LIMIT 1").first()["version"]
    rows_before = spark.table(target).count()

    refused, reason = False, None
    try:
        again = spark.read.options(**CSV_OPTIONS).csv(p["source_file"])
        again.writeTo(target).using("delta").create()
    except Exception as e:
        refused, reason = True, " ".join(str(e).split())[:160]

    version_after = spark.sql(f"DESCRIBE HISTORY {target} LIMIT 1").first()["version"]
    rows_after = spark.table(target).count()

    assert refused, f"{target}: CREATE succeeded on an existing table"
    assert (version_before, rows_before) == (version_after, rows_after), f"{target}: table changed"
    print(f"Create guard OK for {target}: refused ({reason}); version {version_before}, rows {rows_before:,} unchanged")


if DRY_RUN:
    print("DRY RUN - nothing was written, no table or schema was created.")
else:
    target_catalog, target_schema = TARGET.split(".")
    if spark.sql(f"SHOW SCHEMAS IN {target_catalog} LIKE '{target_schema}'").count() == 0:
        spark.sql(f"CREATE SCHEMA {TARGET}")
        print(f"Created schema {TARGET}")
    ensure_log_table()

    run = {
        "run_id": str(uuid.uuid4()),
        "started_at": now(),
        "user": spark.sql("SELECT current_user()").first()[0],
    }
    print(f"Run {run['run_id']} by {run['user']}")

    results = {}
    # smallest sources first, so problems surface before the large files are read
    for p in sorted(plan, key=lambda x: x["source_bytes"] or 0):
        if p["action"] in ("CREATE", "REBUILD"):
            print(f"{p['action']}: {p['target_table']}  <-  {p['source_file']}")
            results[p["table"]] = ingest_table(run, p)
        elif p["action"] == "SKIP_EXISTS":
            write_log(run, p, "SKIPPED", now(), message="table already exists; not modified")
            results[p["table"]] = "SKIPPED"
        elif p["action"] == "EXCLUDED":
            write_log(run, p, "EXCLUDED", now(), message=EXCLUDED_TABLES[p["table"]])
            results[p["table"]] = "EXCLUDED"
        else:  # NO_SOURCE
            write_log(run, p, "FAILED", now(), message="no source file found for this table")
            results[p["table"]] = "FAILED"

    # final coverage check against the registry: expected / excluded / present / missing
    exists_now = {p["table"]: spark.catalog.tableExists(p["target_table"]) for p in plan}
    coverage_report(plan, exists_now, ingested=[t for t, s in results.items() if s == "SUCCESS"])

    if CHECK_CREATE_GUARD:
        created = sorted(
            [p for p in plan if p["action"] == "CREATE" and results.get(p["table"]) == "SUCCESS"],
            key=lambda x: x["source_bytes"] or 0
        )
        if created:
            check_create_guard(created[0])
        else:
            print("Create guard check skipped: no table was created in this run")

    failed = {t: s for t, s in results.items() if s in ("FAILED", "VALIDATION_FAILED")}
    if failed:
        raise RuntimeError(f"Ingestion finished with problems: {failed}")